# Flux Utility Solutions - Load Seed Data

**Purpose**: Load production seed data from Git repository into Snowflake

This notebook loads all reference, operational, and sample data from parquet files stored in the Git repository. Works with Snowflake Git Integration for seamless deployment.

## Prerequisites

1. Git Repository connected to Snowflake (see `scripts/00_git_integration.sql`)
2. Database and tables created (run `01_full_deployment.ipynb` first)
3. ACCOUNTADMIN or appropriate privileges

---

In [ ]:
# Configuration
DATABASE = "FLUX_DEMO"          # Target database
SCHEMA = "PRODUCTION"           # Target schema  
GIT_REPO = "FLUX_REPO"          # Git repository name in Snowflake
BRANCH = "main"                 # Git branch

print(f"Loading seed data into: {DATABASE}.{SCHEMA}")
print(f"From Git repository: {GIT_REPO} ({BRANCH})")

In [ ]:
# Get Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Verify connection
result = session.sql("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect()
print(f"User: {result[0][0]}")
print(f"Role: {result[0][1]}")
print(f"Warehouse: {result[0][2]}")

In [ ]:
# Set context
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
print(f"Context set: {DATABASE}.{SCHEMA}")

## Data Manifest

| Category | Table | Rows | Description |
|----------|-------|------|-------------|
| **Reference** | SUBSTATIONS | 275 | Grid substations |
| | CIRCUIT_METADATA | 8,842 | Distribution circuits |
| | TRANSFORMER_METADATA | 91,554 | Distribution transformers |
| | GRID_POLES_INFRASTRUCTURE | 62,038 | Pole infrastructure |
| | HOUSTON_WEATHER_HOURLY | 4,464 | Weather data |
| | ERCOT_LMP_HOUSTON_ZONE | 45,213 | Energy pricing |
| | POWER_QUALITY_READINGS | 10,000 | PQ events |
| **Operational** | SAP_WORK_ORDERS | 250,488 | Maintenance work orders |
| | OUTAGE_EVENTS | 34,252 | Historical outages |
| **Samples** | METER_INFRASTRUCTURE | 10,000 | Smart meters (sample) |
| | CUSTOMERS_MASTER_DATA | 11,849 | Customers (sample) |

---

## Phase 1: Create Tables

In [ ]:
# Reference Tables DDL
tables_ddl = {
    'SUBSTATIONS': """
        CREATE TABLE IF NOT EXISTS SUBSTATIONS (
            SUBSTATION_ID VARCHAR, SUBSTATION_NAME VARCHAR,
            LATITUDE FLOAT, LONGITUDE FLOAT, CAPACITY_MW FLOAT,
            VOLTAGE_LEVEL_KV FLOAT, INSTALL_DATE DATE, STATUS VARCHAR, REGION VARCHAR
        )
    """,
    'CIRCUIT_METADATA': """
        CREATE TABLE IF NOT EXISTS CIRCUIT_METADATA (
            CIRCUIT_ID VARCHAR, CIRCUIT_NAME VARCHAR, SUBSTATION_ID VARCHAR,
            VOLTAGE_CLASS VARCHAR, CIRCUIT_TYPE VARCHAR, TOTAL_CUSTOMERS NUMBER,
            TOTAL_TRANSFORMERS NUMBER, LINE_MILES FLOAT, INSTALL_DATE DATE,
            LATITUDE FLOAT, LONGITUDE FLOAT, STATUS VARCHAR
        )
    """,
    'TRANSFORMER_METADATA': """
        CREATE TABLE IF NOT EXISTS TRANSFORMER_METADATA (
            TRANSFORMER_ID VARCHAR, TRANSFORMER_NAME VARCHAR, CIRCUIT_ID VARCHAR,
            SUBSTATION_ID VARCHAR, KVA_RATING FLOAT, VOLTAGE_PRIMARY FLOAT,
            VOLTAGE_SECONDARY FLOAT, PHASE_CONFIG VARCHAR, INSTALL_DATE DATE,
            MANUFACTURER VARCHAR, LATITUDE FLOAT, LONGITUDE FLOAT, STATUS VARCHAR,
            LOAD_FACTOR FLOAT, LAST_MAINTENANCE_DATE DATE
        )
    """,
    'GRID_POLES_INFRASTRUCTURE': """
        CREATE TABLE IF NOT EXISTS GRID_POLES_INFRASTRUCTURE (
            POLE_ID VARCHAR, POLE_TYPE VARCHAR, MATERIAL VARCHAR, HEIGHT_FT NUMBER,
            INSTALL_DATE DATE, CIRCUIT_ID VARCHAR, LATITUDE FLOAT, LONGITUDE FLOAT,
            CONDITION_STATUS VARCHAR, LAST_INSPECTION_DATE DATE
        )
    """,
    'HOUSTON_WEATHER_HOURLY': """
        CREATE TABLE IF NOT EXISTS HOUSTON_WEATHER_HOURLY (
            TIMESTAMP TIMESTAMP_NTZ, TEMPERATURE_F FLOAT, HUMIDITY_PCT FLOAT,
            WIND_SPEED_MPH FLOAT, PRECIPITATION_IN FLOAT, WEATHER_CONDITION VARCHAR,
            HEAT_INDEX FLOAT, WIND_CHILL FLOAT
        )
    """,
    'ERCOT_LMP_HOUSTON_ZONE': """
        CREATE TABLE IF NOT EXISTS ERCOT_LMP_HOUSTON_ZONE (
            TIMESTAMP TIMESTAMP_NTZ, LMP_PRICE FLOAT, ENERGY_PRICE FLOAT,
            CONGESTION_PRICE FLOAT, LOSS_PRICE FLOAT, ZONE VARCHAR
        )
    """,
    'POWER_QUALITY_READINGS': """
        CREATE TABLE IF NOT EXISTS POWER_QUALITY_READINGS (
            READING_ID VARCHAR, METER_ID VARCHAR, TIMESTAMP TIMESTAMP_NTZ,
            VOLTAGE FLOAT, FREQUENCY FLOAT, THD_VOLTAGE FLOAT, THD_CURRENT FLOAT,
            POWER_FACTOR FLOAT, SAG_EVENT BOOLEAN, SWELL_EVENT BOOLEAN
        )
    """,
    'SAP_WORK_ORDERS': """
        CREATE TABLE IF NOT EXISTS SAP_WORK_ORDERS (
            WORK_ORDER_ID VARCHAR, WORK_ORDER_TYPE VARCHAR, PRIORITY VARCHAR,
            STATUS VARCHAR, CUSTOMER_ID VARCHAR, DESCRIPTION VARCHAR,
            CREATED_DATE TIMESTAMP_NTZ, SCHEDULED_DATE TIMESTAMP_NTZ,
            COMPLETED_DATE TIMESTAMP_NTZ, CREW_ID VARCHAR,
            ESTIMATED_DURATION_HOURS FLOAT, ACTUAL_DURATION_HOURS FLOAT,
            LABOR_COST FLOAT, PARTS_COST FLOAT
        )
    """,
    'OUTAGE_EVENTS': """
        CREATE TABLE IF NOT EXISTS OUTAGE_EVENTS (
            OUTAGE_ID VARCHAR, TRANSFORMER_ID VARCHAR, CIRCUIT_ID VARCHAR,
            OUTAGE_START_TIME TIMESTAMP_NTZ, OUTAGE_END_TIME TIMESTAMP_NTZ,
            OUTAGE_CAUSE VARCHAR, CUSTOMERS_AFFECTED NUMBER,
            WEATHER_RELATED BOOLEAN, RESTORATION_CREW VARCHAR
        )
    """,
    'METER_INFRASTRUCTURE': """
        CREATE TABLE IF NOT EXISTS METER_INFRASTRUCTURE (
            METER_ID VARCHAR, METER_LATITUDE FLOAT, METER_LONGITUDE FLOAT,
            COMMISSIONED_DATE DATE, METER_TYPE VARCHAR, CUSTOMER_SEGMENT_ID VARCHAR,
            POLE_ID VARCHAR, CIRCUIT_ID VARCHAR, TRANSFORMER_ID VARCHAR,
            SUBSTATION_ID VARCHAR, POLE_TYPE VARCHAR, POLE_MATERIAL VARCHAR,
            POLE_HEIGHT_FT NUMBER, CONDITION_STATUS VARCHAR, ZIP_CODE VARCHAR,
            CITY VARCHAR, COUNTY_NAME VARCHAR, HEALTH_SCORE FLOAT
        )
    """,
    'CUSTOMERS_MASTER_DATA': """
        CREATE TABLE IF NOT EXISTS CUSTOMERS_MASTER_DATA (
            CUSTOMER_ID VARCHAR, FIRST_NAME VARCHAR, LAST_NAME VARCHAR,
            FULL_NAME VARCHAR, PRIMARY_METER_ID VARCHAR, CUSTOMER_SEGMENT VARCHAR,
            SERVICE_ADDRESS VARCHAR, SERVICE_COUNTY VARCHAR, PHONE VARCHAR,
            EMAIL VARCHAR, ACCOUNT_STATUS VARCHAR, SERVICE_START_DATE DATE,
            CREATED_AT TIMESTAMP_NTZ, DATA_SOURCE VARCHAR, ZIP_CODE NUMBER, CITY VARCHAR
        )
    """,
}

for table_name, ddl in tables_ddl.items():
    session.sql(ddl).collect()
    print(f"✓ {table_name}")

print(f"\n✓ Created {len(tables_ddl)} tables")

## Phase 2: Load Data from Git Repository

In [ ]:
# Data loading configuration - maps tables to parquet files
LOAD_CONFIG = [
    # Reference data
    ('SUBSTATIONS', 'reference/substations', 275),
    ('CIRCUIT_METADATA', 'reference/circuit_metadata', 8842),
    ('TRANSFORMER_METADATA', 'reference/transformer_metadata', 91554),
    ('GRID_POLES_INFRASTRUCTURE', 'reference/grid_poles', 62038),
    ('HOUSTON_WEATHER_HOURLY', 'reference/houston_weather', 4464),
    ('ERCOT_LMP_HOUSTON_ZONE', 'reference/ercot_lmp', 45213),
    ('POWER_QUALITY_READINGS', 'reference/power_quality', 10000),
    # Operational data
    ('SAP_WORK_ORDERS', 'operational/sap_work_orders', 250488),
    ('OUTAGE_EVENTS', 'operational/outage_events', 34252),
    # Sample data
    ('METER_INFRASTRUCTURE', 'samples/meter_infrastructure_10k', 10000),
    ('CUSTOMERS_MASTER_DATA', 'samples/customers_master_data_10k', 11849),
]

print(f"Will load {len(LOAD_CONFIG)} tables from @{GIT_REPO}/branches/{BRANCH}/seed_data/parquet/")

In [ ]:
# Load all tables from Git repository stage
GIT_STAGE = f"@{DATABASE}.{SCHEMA}.{GIT_REPO}/branches/{BRANCH}/seed_data/parquet"

print("Loading data from Git repository...")
print("=" * 60)

results = []
for table_name, file_prefix, expected_rows in LOAD_CONFIG:
    try:
        # Truncate table first
        session.sql(f"TRUNCATE TABLE IF EXISTS {table_name}").collect()
        
        # Load from Git stage using COPY INTO
        copy_sql = f"""
            COPY INTO {table_name}
            FROM {GIT_STAGE}/{file_prefix}
            FILE_FORMAT = (TYPE = PARQUET)
            MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
        """
        result = session.sql(copy_sql).collect()
        
        # Get row count
        count = session.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]
        status = "✓" if count >= expected_rows * 0.9 else "⚠️"
        results.append((table_name, count, expected_rows, status))
        print(f"{status} {table_name}: {count:,} rows (expected: {expected_rows:,})")
        
    except Exception as e:
        print(f"✗ {table_name}: {str(e)[:60]}")
        results.append((table_name, 0, expected_rows, "✗"))

print("\n" + "=" * 60)
successful = sum(1 for r in results if r[3] == "✓")
print(f"Loaded {successful}/{len(LOAD_CONFIG)} tables successfully")

## Phase 3: Verify Data Integrity

In [ ]:
# Verify referential integrity
print("REFERENTIAL INTEGRITY CHECK")
print("=" * 60)

checks = [
    ("Circuits → Substations", 
     "SELECT COUNT(DISTINCT c.SUBSTATION_ID) FROM CIRCUIT_METADATA c WHERE c.SUBSTATION_ID IN (SELECT SUBSTATION_ID FROM SUBSTATIONS)"),
    ("Transformers → Circuits",
     "SELECT COUNT(DISTINCT t.CIRCUIT_ID) FROM TRANSFORMER_METADATA t WHERE t.CIRCUIT_ID IN (SELECT CIRCUIT_ID FROM CIRCUIT_METADATA)"),
    ("Meters → Transformers",
     "SELECT COUNT(DISTINCT m.TRANSFORMER_ID) FROM METER_INFRASTRUCTURE m WHERE m.TRANSFORMER_ID IN (SELECT TRANSFORMER_ID FROM TRANSFORMER_METADATA)"),
    ("Customers → Meters",
     "SELECT COUNT(DISTINCT c.PRIMARY_METER_ID) FROM CUSTOMERS_MASTER_DATA c WHERE c.PRIMARY_METER_ID IN (SELECT METER_ID FROM METER_INFRASTRUCTURE)"),
]

for check_name, sql in checks:
    try:
        result = session.sql(sql).collect()[0][0]
        print(f"✓ {check_name}: {result:,} linked records")
    except Exception as e:
        print(f"⚠️ {check_name}: {str(e)[:40]}")

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("SEED DATA LOAD COMPLETE")
print("=" * 60)

total_rows = session.sql(f"""
    SELECT SUM(row_count) 
    FROM {DATABASE}.INFORMATION_SCHEMA.TABLES 
    WHERE table_schema = '{SCHEMA}'
""").collect()[0][0]

print(f"Database: {DATABASE}.{SCHEMA}")
print(f"Total rows loaded: {total_rows:,}")
print(f"\nNext steps:")
print(f"  1. Run 03_create_views.ipynb for semantic views")
print(f"  2. Run 04_cortex_services.ipynb for AI setup")
print(f"  3. Deploy Flux Ops Center or Flux Data Forge")

---

## Alternative: Load from Local Stage

If Git integration isn't available, you can upload parquet files to a named stage:

```sql
-- Create stage
CREATE STAGE IF NOT EXISTS SEED_DATA_STAGE;

-- Upload files (run from CLI)
-- PUT file://seed_data/parquet/reference/*.parquet @SEED_DATA_STAGE/reference/;
-- PUT file://seed_data/parquet/operational/*.parquet @SEED_DATA_STAGE/operational/;
-- PUT file://seed_data/parquet/samples/*.parquet @SEED_DATA_STAGE/samples/;

-- Then change GIT_STAGE variable above to:
-- GIT_STAGE = "@SEED_DATA_STAGE"
```